# CineMatch · LightGCN và SASRec trên GPU (Colab)

Notebook này chạy hai mô hình deep learning làm đối thủ mạnh cho các mô hình giải thích được của đề tài, trên **cùng cách chia dữ liệu và cùng bộ chỉ số** với `evaluate.py`:

1. Tune siêu tham số trên tập validation (20% cuối của train mỗi user, không chạm test) → `results/best_deep.json`
2. Huấn luyện lại trên toàn bộ train, đánh giá trên test cùng 9 mô hình cũ → `results/metrics_deep.csv`

**Trước khi chạy:** Runtime → Change runtime type → chọn **GPU** (T4 là đủ). Thời gian dự kiến: tune ~15–25 phút, đánh giá ~5 phút.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Lấy code

Repo public thì chạy thẳng ô dưới. Repo private: tạo token GitHub (Settings → Developer settings → Fine-grained token, quyền *Contents: Read*) rồi đặt `PRIVATE = True`, ô sẽ hỏi token (không lưu vào notebook).

In [ ]:
import os, getpass, subprocess

REPO = "github.com/DucanhDaniel/he-goi-y.git"
BRANCH = "main"
PRIVATE = False

if not os.path.exists("/content/he-goi-y"):
    url = f"https://{getpass.getpass('GitHub token: ')}@{REPO}" if PRIVATE else f"https://{REPO}"
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, "/content/he-goi-y"], check=True)
os.chdir("/content/he-goi-y")
os.environ["PYTHONPATH"] = "/content/he-goi-y/src"
!git log -1 --oneline

In [ ]:
# Colab đã có sẵn torch, numpy, pandas, scipy, scikit-learn; không cần cài thêm.
import sys, torch
print(sys.version.split()[0], "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"

## 2. Tune trên validation

Lưới nhỏ: LightGCN `layers ∈ {1,2,3} × epochs ∈ {100,300}`, SASRec `dropout ∈ {0.2,0.5} × epochs ∈ {50,100,200}`. Chọn theo NDCG@10 trên validation.

In [ ]:
!python -m he_goi_y.tune --deep
!cat results/best_deep.json

## 3. Đánh giá trên test (cùng 9 mô hình cũ)

In [ ]:
!python -m he_goi_y.evaluate --deep

In [ ]:
import pandas as pd

res = pd.read_csv("results/metrics_deep.csv")
res.sort_values("NDCG@10", ascending=False).round(4)

## 4. Tải kết quả về

Đặt hai file vào thư mục `results/` của repo trên máy rồi commit.

In [ ]:
from google.colab import files

files.download("results/metrics_deep.csv")
files.download("results/best_deep.json")